In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")

In [2]:
cpi_valuesMay = pd.read_csv (
    r"C:\Users\r8885426\OneDrive - FRG\Documents\G4L\CPI_Historic_Values_Zindi_May_23.csv"
)

In [3]:


# -----------------------------
# 1. Standardise column names
# -----------------------------
cpi_valuesMay.columns = cpi_valuesMay.columns.str.strip()

# -----------------------------
# 2. Drop Excel artefact columns
# -----------------------------
cpi_valuesMay = cpi_valuesMay.loc[:, ~cpi_valuesMay.columns.str.contains("^Unnamed")]

# -----------------------------
# 3. Ensure required columns exist
# -----------------------------
required_cols = [
    "Month",
    "Category",
    "Value",
    "Percentage Change (From Prior Month)"
]

for col in required_cols:
    if col not in cpi_valuesMay.columns:
        cpi_valuesMay[col] = np.nan

cpi_valuesMay = cpi_valuesMay[required_cols]

# -----------------------------
# 4. Clean category text (VERY important)
# -----------------------------
cpi_valuesMay["Category"] = cpi_valuesMay["Category"].astype(str).str.strip()

# -----------------------------
# 5. Convert data types
# -----------------------------
cpi_valuesMay["Month"] = pd.to_datetime(
    cpi_valuesMay["Month"], 
    dayfirst=True, 
    errors="coerce"
)

cpi_valuesMay["Value"] = pd.to_numeric(
    cpi_valuesMay["Value"], 
    errors="coerce"
)

cpi_valuesMay["Percentage Change (From Prior Month)"] = pd.to_numeric(
    cpi_valuesMay["Percentage Change (From Prior Month)"], 
    errors="coerce"
)

# -----------------------------
# 6. Drop invalid rows
# -----------------------------
cpi_valuesMay = cpi_valuesMay.dropna(
    subset=["Month", "Category", "Value"]
)

# -----------------------------
# 7. Remove duplicates (Month + Category)
# -----------------------------
cpi_valuesMay = cpi_valuesMay.drop_duplicates(
    subset=["Month", "Category"],
    keep="last"
)

# -----------------------------
# 8. Final sanity check
# -----------------------------
print("May CPI cleaned successfully")
print(cpi_valuesMay.info())
print(cpi_valuesMay.head())
print(cpi_valuesMay.columns)


May CPI cleaned successfully
<class 'pandas.DataFrame'>
RangeIndex: 221 entries, 0 to 220
Data columns (total 4 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   Month                                 221 non-null    datetime64[us]
 1   Category                              221 non-null    str           
 2   Value                                 221 non-null    float64       
 3   Percentage Change (From Prior Month)  221 non-null    float64       
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 7.0 KB
None
       Month                          Category  Value  \
0 2023-05-31                      Headline_CPI  109.6   
1 2023-05-31  Food and non-alcoholic beverages  117.7   
2 2023-05-31   Alcoholic beverages and tobacco  110.6   
3 2023-05-31             Clothing and footwear  104.1   
4 2023-05-31             Housing and utilities  104.6   

   Percentage Chang

In [4]:
cpi_valuesMay.columns

Index(['Month', 'Category', 'Value', 'Percentage Change (From Prior Month)'], dtype='str')

In [5]:
cpi_valuesMay.info

<bound method DataFrame.info of          Month                          Category  Value  \
0   2023-05-31                      Headline_CPI  109.6   
1   2023-05-31  Food and non-alcoholic beverages  117.7   
2   2023-05-31   Alcoholic beverages and tobacco  110.6   
3   2023-05-31             Clothing and footwear  104.1   
4   2023-05-31             Housing and utilities  104.6   
..         ...                               ...    ...   
216 2022-01-31                     Communication   99.8   
217 2022-01-31            Recreation and culture  100.2   
218 2022-01-31                         Education  100.0   
219 2022-01-31            Restaurants and hotels  101.2   
220 2022-01-31  Miscellaneous goods and services  100.6   

     Percentage Change (From Prior Month)  
0                                     0.2  
1                                     0.3  
2                                     0.4  
3                                     0.4  
4                                     0

In [6]:
print(cpi_valuesMay["Month"].min())
print(cpi_valuesMay["Month"].max())

cpi_valuesMay.groupby("Category")["Month"].nunique()

2022-01-31 00:00:00
2023-05-31 00:00:00


Category
Alcoholic beverages and tobacco     17
Clothing and footwear               17
Communication                       17
Education                           17
Food and non-alcoholic beverages    17
Headline_CPI                        17
Health                              17
Household contents and services     17
Housing and utilities               17
Miscellaneous goods and services    17
Recreation and culture              17
Restaurants and hotels              17
Transport                           17
Name: Month, dtype: int64

In [7]:

 
cpi_valuesMay = cpi_valuesMay.sort_values(["Category", "Month"])


In [8]:
def nowcast_july(series, window=6, steps_ahead=2, alpha=0.5):
    """
    Dampened linear nowcast:
    alpha = 0 → last value only
    alpha = 1 → full linear trend
    """
    series = series.dropna()

    last_value = series.iloc[-1]

    if len(series) < 3:
        return float(last_value)

    y = series.tail(window).values
    X = np.arange(len(y)).reshape(-1, 1)

    model = LinearRegression()
    model.fit(X, y)

    future_x = np.array([[len(y) + steps_ahead - 1]])
    trend_pred = model.predict(future_x)[0]

    # Blend trend with last observed value
    dampened_pred = alpha * trend_pred + (1 - alpha) * last_value

    return float(dampened_pred)

In [9]:
categories = [
    "Headline_CPI",
    "Food and non-alcoholic beverages",
    "Alcoholic beverages and tobacco",
    "Clothing and footwear",
    "Housing and utilities",
    "Household contents and services",
    "Health",
    "Transport",
    "Communication",
    "Recreation and culture",
    "Education",
    "Restaurants and hotels",
    "Miscellaneous goods and services"
]


In [10]:
predictions = {}

for cat in categories:
    series = (
        cpi_valuesMay[cpi_valuesMay["Category"] == cat]
        .sort_values("Month")["Value"]
    )

    pred = nowcast_july(series)
    predictions[cat] = round(pred, 1)

In [11]:
submission = pd.DataFrame(
    [
        ("July_headline CPI", predictions["Headline_CPI"]),
        ("July_food and non-alcoholic beverages", predictions["Food and non-alcoholic beverages"]),
        ("July_alcoholic beverages and tobacco", predictions["Alcoholic beverages and tobacco"]),
        ("July_clothing and footwear", predictions["Clothing and footwear"]),
        ("July_housing and utilities", predictions["Housing and utilities"]),
        ("July_household contents and services", predictions["Household contents and services"]),
        ("July_health", predictions["Health"]),
        ("July_transport", predictions["Transport"]),
        ("July_communication", predictions["Communication"]),
        ("July_recreation and culture", predictions["Recreation and culture"]),
        ("July_education", predictions["Education"]),
        ("July_restaurants and hotels", predictions["Restaurants and hotels"]),
        ("July_miscellaneous goods and services", predictions["Miscellaneous goods and services"]),
    ],
    columns=["ID", "Value"]
)

submission

,ID,Value
0,July_headline CPI,110.3
1,July_food and non-alcoholic beverages,119.1
2,July_alcoholic beverages and tobacco,111.7
3,July_clothing and footwear,104.3
4,July_housing and utilities,104.7
5,July_household contents and services,108.0
6,July_health,111.8
7,July_transport,113.4
8,July_communication,99.9
9,July_recreation and culture,105.2
